In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum, avg, min, max,
    countDistinct, approx_count_distinct,
    round, first, collect_list, collect_set,
    expr, when
)

spark = SparkSession.builder.appName("PysparkAgg").master("local[*]").getOrCreate()

data = [
    ("2026-03-01", "electronics", "laptop", 1200.00, 3),
    ("2026-03-01", "electronics", "phone", 800.00, 7),
    ("2026-03-01", "clothing", "jacket", 150.00, 12),
    ("2026-03-01", "clothing", "shoes", 90.00, 20),
    ("2026-03-02", "electronics", "laptop", 1200.00, 5),
    ("2026-03-02", "electronics", "tablet", 600.00, 4),
    ("2026-03-02", "clothing", "jacket", 150.00, 8),
    ("2026-03-02", "clothing", "shoes", 90.00, 15),
    ("2026-03-03", "electronics", "phone", 800.00, 6),
    ("2026-03-03", "clothing", "jacket", 150.00, 10),
    ("2026-03-03", "clothing", "shoes", 90.00, 18),
    ("2026-03-03", "electronics", "laptop", 1200.00, 2),
]

df = spark.createDataFrame(
    data,
    schema=["order_date", "category", "product", "unit_price", "quantity"]
)

In [3]:
df.show()

+----------+-----------+-------+----------+--------+
|order_date|   category|product|unit_price|quantity|
+----------+-----------+-------+----------+--------+
|2026-03-01|electronics| laptop|    1200.0|       3|
|2026-03-01|electronics|  phone|     800.0|       7|
|2026-03-01|   clothing| jacket|     150.0|      12|
|2026-03-01|   clothing|  shoes|      90.0|      20|
|2026-03-02|electronics| laptop|    1200.0|       5|
|2026-03-02|electronics| tablet|     600.0|       4|
|2026-03-02|   clothing| jacket|     150.0|       8|
|2026-03-02|   clothing|  shoes|      90.0|      15|
|2026-03-03|electronics|  phone|     800.0|       6|
|2026-03-03|   clothing| jacket|     150.0|      10|
|2026-03-03|   clothing|  shoes|      90.0|      18|
|2026-03-03|electronics| laptop|    1200.0|       2|
+----------+-----------+-------+----------+--------+



In [4]:
# Group by + Agg
df.groupBy("category").agg(
    count("*").alias("total_orders"),
    sum("quantity").alias("total_units"),
    round(avg("unit_price"), 2).alias("avg_price"),
).show()

+-----------+------------+-----------+---------+
|   category|total_orders|total_units|avg_price|
+-----------+------------+-----------+---------+
|electronics|           6|         27|   966.67|
|   clothing|           6|         83|    120.0|
+-----------+------------+-----------+---------+



In [5]:
# Grouping by Multiple Columns
df.groupBy("order_date", "category").agg(
    sum(col("unit_price") * col("quantity")).alias("revenue"),
    sum("quantity").alias("units_sold"),
).orderBy("order_date", "category").show()

+----------+-----------+-------+----------+
|order_date|   category|revenue|units_sold|
+----------+-----------+-------+----------+
|2026-03-01|   clothing| 3600.0|        32|
|2026-03-01|electronics| 9200.0|        10|
|2026-03-02|   clothing| 2550.0|        23|
|2026-03-02|electronics| 8400.0|         9|
|2026-03-03|   clothing| 3120.0|        28|
|2026-03-03|electronics| 7200.0|         8|
+----------+-----------+-------+----------+



In [7]:
# Exact and Approximate counting
df.groupBy("category").agg(
    countDistinct("product").alias("exact_products"),
    approx_count_distinct("product").alias("approx_products"),
).show()

+-----------+--------------+---------------+
|   category|exact_products|approx_products|
+-----------+--------------+---------------+
|electronics|             3|              3|
|   clothing|             2|              2|
+-----------+--------------+---------------+



In [11]:
# Conditional Aggregation
df.groupBy("order_date").agg(
    sum("quantity").alias("total_units"),
    sum(
        when(col("category") == "electronics", col("quantity")).otherwise(0)
    ).alias("electronics_units"),
    sum(
        when(col("category") == "clothing", col("quantity")).otherwise(0)
    ).alias("clothing_units"),
    count(
        when(col("unit_price") > 500, True)
    ).alias("premium_order_count"),
).show()

+----------+-----------+-----------------+--------------+-------------------+
|order_date|total_units|electronics_units|clothing_units|premium_order_count|
+----------+-----------+-----------------+--------------+-------------------+
|2026-03-01|         42|               10|            32|                  2|
|2026-03-02|         32|                9|            23|                  2|
|2026-03-03|         36|                8|            28|                  2|
+----------+-----------+-----------------+--------------+-------------------+



In [12]:
df.groupBy("category").agg(
    collect_set("product").alias("unique_products"),
    collect_list("product").alias("all_products"),
).show(truncate=False)

+-----------+-----------------------+----------------------------------------------+
|category   |unique_products        |all_products                                  |
+-----------+-----------------------+----------------------------------------------+
|electronics|[tablet, laptop, phone]|[laptop, phone, laptop, tablet, phone, laptop]|
|clothing   |[shoes, jacket]        |[jacket, shoes, jacket, shoes, jacket, shoes] |
+-----------+-----------------------+----------------------------------------------+



In [13]:
df.groupBy("order_date").pivot("category").agg(
    sum("quantity").alias("units")
).orderBy("order_date").show()

+----------+--------+-----------+
|order_date|clothing|electronics|
+----------+--------+-----------+
|2026-03-01|      32|         10|
|2026-03-02|      23|          9|
|2026-03-03|      28|          8|
+----------+--------+-----------+



In [14]:
df.groupBy("order_date").pivot(
    "category",
    ["electronics", "clothing"]  # explicit values — avoids extra scan
).agg(
    sum("quantity").alias("units")
).orderBy("order_date").show()

+----------+-----------+--------+
|order_date|electronics|clothing|
+----------+-----------+--------+
|2026-03-01|         10|      32|
|2026-03-02|          9|      23|
|2026-03-03|          8|      28|
+----------+-----------+--------+

